# RAG from Scratch to Production — Episode 12/20
## Context engineering: what goes into the prompt

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JakkulaMahendhar/rag-from-scratch/blob/main/notebooks/12_context_engineering.ipynb)

Retrieval found the chunks. Now we decide **what the LLM actually reads**: how many chunks, in what order, trimmed or not, and how much text in total. That's *context engineering*.

The common instinct is "more context is safer: if the answer is in there somewhere, the LLM will find it". **Let's test that.**

**In this episode:**
1. Send 1, 3, 5 or 20 chunks, and measure the answers.
2. Move the right chunk to the start, middle and end of the context.
3. Try two prompt fixes.
4. Test the [production app](https://github.com/JakkulaMahendhar/rag-chatbot)'s compressor and token counter.

**Needs an LLM:** Gemini (free key) or local Ollama. About 110 LLM calls in total.

## 0. Setup

The usual `generate()`, plus a counter for prompt size, and a pause between calls on Gemini so the free tier's rate limit isn't hit (the whole notebook then takes about 12 minutes).

In [ ]:
%pip install -q google-generativeai requests sentence-transformers

In [1]:
import os, re, time, requests
import numpy as np

PROVIDER = os.getenv("LLM_PROVIDER", "gemini")   # "gemini" or "ollama"
GEMINI_MODEL = "gemini-3.6-flash"
OLLAMA_MODEL = "llama3.1"
SECONDS_BETWEEN_CALLS = 6 if PROVIDER == "gemini" else 0   # stay under the free tier's rate limit


def _gemini_key():
    try:
        from google.colab import userdata  # Colab secrets
        return userdata.get("GEMINI_API_KEY")
    except Exception:
        return os.getenv("GEMINI_API_KEY") or __import__("getpass").getpass("GEMINI_API_KEY: ")


if PROVIDER == "gemini":
    import google.generativeai as genai
    genai.configure(api_key=_gemini_key())
    _model = genai.GenerativeModel(GEMINI_MODEL)

PROMPT_TOKENS = []   # prompt size of each call


def generate(prompt: str) -> str:
    """One interface, two providers. Temperature 0 so results are repeatable."""
    time.sleep(SECONDS_BETWEEN_CALLS)
    if PROVIDER == "gemini":
        response = _model.generate_content(prompt, generation_config={"temperature": 0})
        PROMPT_TOKENS.append(response.usage_metadata.prompt_token_count)
        return response.text.strip()
    r = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False,
              "options": {"temperature": 0}},
        timeout=300,
    )
    r.raise_for_status()
    PROMPT_TOKENS.append(r.json().get("prompt_eval_count", 0))
    return r.json()["response"].strip()


generate("Say OK")   # warm-up
print(f"Using {PROVIDER}")

Using ollama


## 1. The documents and the test

20 chunks: Episode 10's handbook plus its "hard" handbook, full of near-twins (full-time vs part-time leave, birth vs adoptive parents, domestic vs international trips).

10 questions, each with the words a correct answer must contain. An answer counts as correct only if it contains one of them **and** isn't a refusal.

In [2]:
CHUNKS = [
    "Remote Work\nEmployees may work remotely up to 2 days per week. Wednesday is a mandatory in-office day for all teams. Remote days must be logged in the HR portal by Friday of the previous week.",
    "Annual Leave\nFull-time employees receive 24 days of paid annual leave per calendar year. A maximum of 5 unused days may be carried over into the next year. Book leave with form HR-12.",
    "Parental Leave\nNew parents receive 16 weeks of fully paid leave. Submit form HR-17 at least 8 weeks before the expected start date.",
    "Expenses\nThe meal allowance during business travel is capped at 40 USD per day. Travel expenses are claimed with form EXP-22 within 30 days. Claims above 500 USD need the finance director.",
    "Learning Budget\nEvery employee has an annual learning budget of 1,200 USD for courses and conferences. It does not roll over. Requests are approved by your direct manager.",
    "Equipment\nEvery employee receives a ThinkPad X1 Carbon laptop, refreshed every 3 years. Lost or stolen devices must be reported to IT within 24 hours.",
    "IT Support\nIf the VPN shows error E-4031, your access token has expired: reset it in the Okta portal. For error E-4032, contact the IT helpdesk.",
    "Security\nMulti-factor authentication is mandatory on all company accounts. Passwords must be at least 14 characters long and must never be reused.",
    "Annual Leave: Full-time Employees\nFull-time employees receive 24 days of paid annual leave per year. Up to 5 unused days may be carried over into the next year.",
    "Annual Leave: Part-time Employees\nPart-time employees receive 12 days of paid annual leave per year. Unused days cannot be carried over.",
    "Parental Leave: Birth Parents\nBirth parents receive 16 weeks of fully paid leave. Submit form HR-17 at least 8 weeks before the expected due date.",
    "Parental Leave: Adoptive Parents\nAdoptive parents receive 10 weeks of fully paid leave, starting from the day the child joins the family. Submit form HR-18.",
    "Travel Expenses: Domestic Trips\nOn trips inside the country, the meal allowance is 40 USD per day. Hotels are booked through the travel portal.",
    "Travel Expenses: International Trips\nOn trips abroad, the meal allowance is 75 USD per day. International flights over 6 hours may be booked in premium economy.",
    "Remote Work: Office Staff\nOffice staff may work remotely up to 2 days per week. Wednesday is a mandatory in-office day.",
    "Remote Work: Field Engineers\nField engineers work fully remotely and visit the office once per quarter for planning.",
    "Laptops: Engineering\nEngineers receive a ThinkPad X1 Carbon, replaced every 3 years.",
    "Laptops: Sales\nSales staff receive a MacBook Air, replaced every 4 years.",
    "Learning Budget\nEvery employee has an annual learning budget of 1,200 USD for courses and conferences. It does not roll over.",
    "Security\nMulti-factor authentication is mandatory. Passwords must be at least 14 characters long and must never be reused.",
]

TESTS = [
    # (question, the section that answers it, words a correct answer contains)
    ("How many leave days do part-time employees get?",     "Annual Leave: Part-time Employees",    ["12"]),
    ("Can part-timers carry unused leave into next year?",  "Annual Leave: Part-time Employees",    ["cannot", "not", "no"]),
    ("How much paid leave do I get if I adopt a child?",    "Parental Leave: Adoptive Parents",     ["10"]),
    ("What's the meal allowance on a trip to Germany?",     "Travel Expenses: International Trips", ["75"]),
    ("What laptop does the sales team get?",                "Laptops: Sales",                       ["macbook"]),
    ("How often do field engineers come to the office?",    "Remote Work: Field Engineers",         ["quarter"]),
    ("What does VPN error E-4031 mean?",                    "IT Support",                           ["token", "expired"]),
    ("Who approves claims above 500 USD?",                  "Expenses",                             ["finance director"]),
    ("How many characters must a password have?",           "Security",                             ["14"]),
    ("Which form do birth parents submit?",                 "Parental Leave: Birth Parents",        ["hr-17"]),
]

FALLBACK = "I don't have enough information from the provided documents."

def title(chunk: str) -> str:
    return chunk.split("\n")[0]

def is_correct(answer: str, must_contain: list[str]) -> bool:
    a = answer.lower()
    return any(word in a for word in must_contain) and "don't have enough information" not in a

The prompt is Episode 1's grounding prompt, with numbered sources:

In [3]:
PLAIN_RULES = f'Answer only using the context below. If the answer is not in the context, respond exactly: "{FALLBACK}"'

def ask(question: str, chunks: list[str], rules: str = PLAIN_RULES) -> str:
    context = "\n\n".join(f"SOURCE {i + 1}:\n{c}" for i, c in enumerate(chunks))
    return generate(f"""{rules}

Context:
{context}

Question: {question}
Answer:""")

## 2. Experiment 1: how many chunks?

Rank all 20 chunks by embedding similarity (Episode 5), and send the top 1, 3, 5 or all 20. The production app sends its top 3 after reranking.

In [4]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
vectors = embedder.encode(CHUNKS, normalize_embeddings=True)

def top_chunks(question: str, k: int) -> list[str]:
    order = np.argsort(-(vectors @ embedder.encode(question, normalize_embeddings=True)))
    return [CHUNKS[i] for i in order[:k]]


def run(label: str, build_context, rules: str = PLAIN_RULES):
    PROMPT_TOKENS.clear()
    correct, failures = 0, []
    for question, section, must_contain in TESTS:
        answer = ask(question, build_context(question, section), rules)
        if is_correct(answer, must_contain):
            correct += 1
        else:
            failures.append((question, answer[:70]))
    print(f"{label:<28} {correct}/10 correct   ~{int(np.mean(PROMPT_TOKENS))} prompt tokens")
    for question, answer in failures:
        print(f"      ❌ {question:<52} → {answer!r}")
    return correct


K_RESULTS = {k: run(f"top {k} chunk{'s' if k > 1 else ''}", lambda q, s, k=k: top_chunks(q, k)) for k in [1, 3, 5, 20]}

top 1 chunk                  10/10 correct   ~91 prompt tokens


top 3 chunks                 8/10 correct   ~164 prompt tokens
      ❌ Can part-timers carry unused leave into next year?   → "I don't have enough information from the provided documents."
      ❌ What's the meal allowance on a trip to Germany?      → "I don't have enough information from the provided documents."


top 5 chunks                 8/10 correct   ~239 prompt tokens
      ❌ Can part-timers carry unused leave into next year?   → "I don't have enough information from the provided documents."
      ❌ What's the meal allowance on a trip to Germany?      → "I don't have enough information from the provided documents."


top 20 chunks                9/10 correct   ~767 prompt tokens
      ❌ What's the meal allowance on a trip to Germany?      → "I don't have enough information from the provided documents."


More context did **not** help. With just the best chunk, every answer was right. Add similar-looking chunks and some answers fail, while the prompt grows.

And look at **how** they fail: not wrong answers, but **refusals**. The answer is right there in the prompt, yet the model says *"I don't have enough information"*. Here's one up close:

In [5]:
question = "Can part-timers carry unused leave into next year?"
for chunk in top_chunks(question, 3):
    print("SOURCE:", title(chunk))
print("\nAnswer:", ask(question, top_chunks(question, 3)))

SOURCE: Annual Leave: Part-time Employees
SOURCE: Annual Leave: Full-time Employees
SOURCE: Annual Leave


Answer: I don't have enough information from the provided documents.


The part-time section says *"Unused days cannot be carried over"*. But right next to it, the full-time section says *"Up to 5 unused days **may** be carried over"*. Faced with two sources that seem to disagree, and a prompt that says "only answer from the context", the model plays it safe and refuses.

Extra context didn't make the bot lie. **It made it give up.**

## 3. Experiment 2: does position matter?

Research on long prompts found that LLMs use information at the **start and end** of the context better than in the middle ("lost in the middle"). Let's put all 20 chunks in, with the right one first, in the middle, or last:

In [6]:
rng = np.random.default_rng(0)

def place_right_chunk(position: int):
    def build(question, section):
        right = next(c for c in CHUNKS if title(c) == section)
        others = [c for c in CHUNKS if c != right]
        rng.shuffle(others)
        return others[:position - 1] + [right] + others[position - 1:]
    return build

for position in [1, 10, 20]:
    run(f"right chunk at position {position}", place_right_chunk(position))

right chunk at position 1    9/10 correct   ~767 prompt tokens
      ❌ What's the meal allowance on a trip to Germany?      → "I don't have enough information from the provided documents."


right chunk at position 10   8/10 correct   ~767 prompt tokens
      ❌ Can part-timers carry unused leave into next year?   → "I don't have enough information from the provided documents."
      ❌ What's the meal allowance on a trip to Germany?      → "I don't have enough information from the provided documents."


right chunk at position 20   7/10 correct   ~767 prompt tokens
      ❌ Can part-timers carry unused leave into next year?   → "I don't have enough information from the provided documents."
      ❌ How much paid leave do I get if I adopt a child?     → "I don't have enough information from the provided documents."
      ❌ What's the meal allowance on a trip to Germany?      → "I don't have enough information from the provided documents."


In my local run, the score slipped a little as the right chunk moved later (9 → 8 → 7), but that's not the "dip in the middle" pattern, and the other 19 chunks were also shuffled differently each time. On 10 questions, a 2-question gap is weak evidence. "Lost in the middle" is a long-context effect (tens of thousands of tokens), and our prompts are under 1,000. Every failure here is the same near-twin refusal again: **what** you include matters more than where.

## 4. Experiment 3: can the prompt fix the refusals?

Two ideas:
1. **Explain the twins**: tell the model sources may cover different groups, and to pick the matching one.
2. **Name the source first**: make it commit to one source before answering.

In [7]:
GROUPS_RULES = ("Answer only using the context below. Sources can describe different groups of people or different "
                "situations (for example full-time vs part-time staff, or domestic vs international trips). Work out which "
                f'source matches the question and answer from it. If none of them answers it, respond exactly: "{FALLBACK}"')

NAME_SOURCE_RULES = ("Answer only using the context below.\n"
                     "First write the title of the one source that matches the question (sources may cover different groups "
                     "of people or situations), then answer from that source in one sentence.\n"
                     f'If no source answers the question, respond exactly: "{FALLBACK}"')

for k in [3, 5]:
    run(f"top {k}, explain the twins", lambda q, s, k=k: top_chunks(q, k), GROUPS_RULES)
    run(f"top {k}, name the source", lambda q, s, k=k: top_chunks(q, k), NAME_SOURCE_RULES)

top 3, explain the twins     8/10 correct   ~200 prompt tokens
      ❌ Can part-timers carry unused leave into next year?   → "I don't have enough information from the provided documents."
      ❌ What's the meal allowance on a trip to Germany?      → "I don't have enough information from the provided documents."


top 3, name the source       9/10 correct   ~194 prompt tokens
      ❌ Can part-timers carry unused leave into next year?   → "I don't have enough information from the provided documents."


top 5, explain the twins     9/10 correct   ~275 prompt tokens
      ❌ What's the meal allowance on a trip to Germany?      → "I don't have enough information from the provided documents."


top 5, name the source       10/10 correct   ~269 prompt tokens


Compare with the plain prompt in Experiment 1 (8/10 at both sizes). In my local run (llama3.1):
- **Explaining the twins** barely helped: +0 and +1.
- **Naming the source first** helped more: +1 and +2, reaching 10/10 with 5 chunks. Committing to one source seems to stop the model from freezing between two that look contradictory.

Promising, but not proven: these are differences of 1–2 questions out of 10, and in an earlier test run the same "name the source" prompt **lost** a question when given only 1 chunk. Test it on your own questions before relying on it.

The most reliable lever was still **sending fewer, better chunks**. Prompt wording comes second.

| | Plain | Explain the twins | Name the source |
|---|---|---|---|
| Top 3 chunks | 8/10 | 8/10 | 9/10 |
| Top 5 chunks | 8/10 | 9/10 | **10/10** |

## 5. The production compressor

Before building the prompt, the [production app](https://github.com/JakkulaMahendhar/rag-chatbot) compresses each chunk (`app/services/context_compressor.py`): it keeps only the sentences that contain a word from the question (any word longer than 2 letters). Does it ever cut out the answer?

In [8]:
def compress(question: str, chunk: str) -> str:
    """Production's logic: keep sentences containing any question word longer than 2 letters."""
    keywords = [w.lower() for w in re.findall(r"\w+", question) if len(w) > 2]
    sentences = re.split(r"(?<=[.!?])\s+", chunk)
    relevant = [s for s in sentences if any(k in s.lower() for k in keywords)]
    return "\n".join(relevant) if relevant else chunk


lost = 0
for question, section, must_contain in TESTS:
    chunk = next(c for c in CHUNKS if title(c) == section)
    compressed = compress(question, chunk)
    kept = any(w in compressed.lower() for w in must_contain)
    lost += not kept
    print(f"{'kept' if kept else 'LOST'}  {len(chunk):>3} → {len(compressed):>3} chars   {question}")
print(f"\nAnswers cut out by compression: {lost}/10")

kept  136 → 136 chars   How many leave days do part-time employees get?
kept  136 → 136 chars   Can part-timers carry unused leave into next year?
kept  156 → 137 chars   How much paid leave do I get if I adopt a child?
kept  160 →  91 chars   What's the meal allowance on a trip to Germany?
kept   73 →  73 chars   What laptop does the sales team get?
kept  116 → 116 chars   How often do field engineers come to the office?
kept  144 → 144 chars   What does VPN error E-4031 mean?
kept  188 → 127 chars   Who approves claims above 500 USD?
kept  146 →  71 chars   How many characters must a password have?
kept  146 → 146 chars   Which form do birth parents submit?
Answers cut out by compression: 0/10


It never cut out an answer here, and it saved little: short chunks are mostly kept whole.

But see what it does to a follow-up question. Production compresses with the **original** question, not the rewritten one (Episode 9):

In [9]:
learning_budget = CHUNKS[4]
print("Chunk:     ", repr(learning_budget))
print("Compressed:", repr(compress("Does it roll over?", learning_budget)))

Chunk:      'Learning Budget\nEvery employee has an annual learning budget of 1,200 USD for courses and conferences. It does not roll over. Requests are approved by your direct manager.'
Compressed: 'It does not roll over.'


Only *"It does not roll over."* survives. The sentence that says what "it" is, the learning budget, is gone. The fact is kept, but its **subject** is lost, the same problem small chunks had in Episode 4.

## 6. Counting tokens

The production app keeps the context under a budget of 3,000 tokens (4,000 minus 1,000 reserved for the answer). But it counts tokens as **words**:

In [10]:
all_text = "\n\n".join(CHUNKS)
words = len(all_text.split())
PROMPT_TOKENS.clear()
generate(all_text)
print(f"words: {words}   real {PROVIDER} tokens: {PROMPT_TOKENS[-1]}   ratio {PROMPT_TOKENS[-1] / words:.2f}")

words: 476   real ollama tokens: 641   ratio 1.35


Real tokens run about a third higher than words for English text, more for codes and numbers. So a "3,000-token" budget is really around 4,000 tokens. That's harmless with today's large context windows, but if you ever set a tight budget, count with the model's real tokenizer.

## 7. What production does

The [production app](https://github.com/JakkulaMahendhar/rag-chatbot):
- Sends the **top 3** chunks after reranking (Episode 11 drops any below 0.20 first). Experiment 1 suggests that when one chunk is clearly best, sending only that one avoids confusing near-twins.
- **Compresses** with the original question, so follow-ups can lose their subject.
- Counts the token budget in **words**.
- Labels each chunk with *SOURCE / Document / Chunk ID / Similarity Score*. Those labels once leaked into answers (bug #29 in its bug log), fixed with an explicit "don't repeat these labels" rule.

## Recap

| Myth | What we measured |
|---|---|
| "More context is safer" | 1 chunk: all correct. Add near-twins: the model **refuses** answers that are right there |
| "Put the answer at the start" | At this size, position barely matters |
| "A better prompt fixes it" | Naming the source first helped (+1–2 questions): promising, not proven |
| "Compression saves tokens" | Little saving on short chunks, and follow-ups can lose their subject |

**The rule:** context quality beats context quantity. Send the **fewest** chunks that contain the answer, especially when your documents have look-alike sections.

**Next, Episode 13: prompt building.** The instructions around the context: grounding rules, refusals, and the labels that leaked.

⭐ Notebooks: https://github.com/JakkulaMahendhar/rag-from-scratch
🏗️ Full production app: https://github.com/JakkulaMahendhar/rag-chatbot
📸 Instagram: [@codingwithmahi](https://instagram.com/codingwithmahi)